# Análise Exploratória de Dados Multivariada - Dados de Gênero

O código apresentado neste notebook realiza uma análise exploratória entre as variáveis presentes na base de dados e tem como objetivo identificar padrões e verificar a viabilidade da base de dados para o contexto de justiça algorítimica. 

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from scipy import stats

import statsmodels.formula.api as smf

In [2]:
df = pd.read_csv("../dataset/dataset_limpo_multivariado.csv")

Há alguns dados sensíveis principais que podem ser usados para uma investigação de justiça algorítmica. Na nossa base de dados eles são: 

- Idade
- Grau de instrução escolar
- Status civil
- Status econômico (tendo como proxy residência)
- Sexo biológico 

Abaixo vou focar em realizar uma exploração estatística entre as features/variaveis com relação ao gênero dos clientes, o objetivo de verificar se possuímos correlações suficientes que justifiquem um aprofundamento em situações de (in)justiça algorítimica. As variáveis que serão correlacionadas são:

- Tipo de proposta aprovada
- Valor proposto no contrato
- Valor aprovado no contrato
- Tempo máximo de atraso de uma parcela
- Tempo total de atraso contando todas as parcelas
- Valor das prestações
- Valor de juros aprovado
- Taxa de juros aprovado
- Histórico no SPC, SERASA e BNB

Não são as variáveis mais importantes, mas pode ter alguma utilidade investigar as variáveis de:

- Valor do seguro

Para facilitar as analises, vou criar uma coluna que indica a porcentagem de mulheres dentro daquele grupo de clientes

In [3]:
df['PORCENTAGEM_MULHERES_GRUPO'] = (df['QT_CLI_MLH']/df['QT_CLI']).round(1)

E também vou criar uma coluna categória que indique se aquele contrato diz respeito a um conjunto de clientes composto exclusivamente por mulheres, exclusivamente por homens ou se é misto.

In [4]:
condicoes = [df['PORCENTAGEM_MULHERES_GRUPO'] == 0,
    (df['PORCENTAGEM_MULHERES_GRUPO'] > 0) & (df['PORCENTAGEM_MULHERES_GRUPO'] < 1),
    df['PORCENTAGEM_MULHERES_GRUPO'] == 1,
]
nomes = ['Apenas Homens', 'Misto', 'Apenas Mulheres']

df['PERFIL_GENERO_GRUPO_CLIENTES'] = np.select(condicoes, nomes, default='Indefinido')

Acredito que possa ser mais interessante realizar uma separação do meu dataset em três grandes grupos demográficos: (1) o grupo em que devedor seja composto apenas por grupo de mulheres, (2) o grupo devedor seja composto apenas por grupo de homens e (3) o grupo devedor é um grupo misto em gênero.

In [5]:
df_apenas_mulheres = df[df['PERFIL_GENERO_GRUPO_CLIENTES'] == 'Apenas Mulheres']
df_apenas_homens = df[df['PERFIL_GENERO_GRUPO_CLIENTES'] == 'Apenas Homens']
df_misto = df[df['PERFIL_GENERO_GRUPO_CLIENTES'] == 'Misto']

In [6]:
print(f'Há {len(df_apenas_mulheres)} linhas/registros de grupo de clientes composto apenas por mulheres')
print(f'Há {len(df_apenas_homens)} linhas/registros de grupos de clientes composto apenas por homens')
print(f'Há {len(df_misto)} linhas/registros de grupo misto em gênero composto')

Há 83725 linhas/registros de grupo de clientes composto apenas por mulheres
Há 19974 linhas/registros de grupos de clientes composto apenas por homens
Há 177837 linhas/registros de grupo misto em gênero composto


Observamos que em maioria os grupos de pessoas devedoras é homogêneo (177.837 grupos), seguido por grupos exclusivo de mulheres (83.725 grupos). É evidente portanto que há um desbalancemanto na base de dados, vou investigar melhor como as demais variáveis estão distribuidas considerando essa separação inicial dos grupos.

In [7]:
media_mulheres = df_apenas_mulheres['QT_CLI'].mean()
media_homens = df_apenas_homens['QT_CLI'].mean()
media_misto = df_misto['QT_CLI'].mean()

mediana_mulheres = df_apenas_mulheres['QT_CLI'].median()
mediana_homens = df_apenas_homens['QT_CLI'].median()
mediana_misto = df_misto['QT_CLI'].median()

print(f'Há em média {media_mulheres:.0f} mulheres por contrato, no grupo de clientes exclusivo de mulheres')
print(f'Há em média {media_homens:.0f} homens por contrato, no grupo de clientes exclusivo de homens')
print(f'Há em média {media_misto:.0f} pessoas por contrato, no grupo de clientes misto em gênero')
print("\n")
print(f'A mediana é de {mediana_mulheres:.0f} mulheres por contrato, no grupo de clientes exclusivo de mulheres')
print(f'A mediana é de {mediana_homens:.0f} homens por contrato, no grupo de clientes exclusivo de homens')
print(f'A mediana é de {mediana_misto:.0f} pessoas por contrato, no grupo de clientes misto em gênero')

Há em média 4 mulheres por contrato, no grupo de clientes exclusivo de mulheres
Há em média 4 homens por contrato, no grupo de clientes exclusivo de homens
Há em média 4 pessoas por contrato, no grupo de clientes misto em gênero


A mediana é de 4 mulheres por contrato, no grupo de clientes exclusivo de mulheres
A mediana é de 3 homens por contrato, no grupo de clientes exclusivo de homens
A mediana é de 4 pessoas por contrato, no grupo de clientes misto em gênero


Arredondando as casas decimais (afinal não existe meia pessoa), observamos que a mediana da quantidade de pessoas por contrato/empréstimo é menor entre os homens.

Para as realizar as analises abaixo, vou realizar uma análise entreos três grupos para que eu consiga ter um contexto do comportamento do banco e dos clientes. Nos próximos notebooks, vou tentar sempre que conveniente manter separado em (1) análise comparativa e de _fairness_ entre os extremos - grupos de apenas mulheres e grupos de apenas homens, (2) análise interna do grupo misto de clientes. Acho que isso vai me ajudar a evitar que os extremos enviesem análises de correlações entre variáveis. 

#### Análise sobre o tipo de proposta

In [8]:
tabela_tipo_proposta_por_grupo = (pd.crosstab(df['PERFIL_GENERO_GRUPO_CLIENTES'], df['ID_TP_PRP'], normalize='index')*100).round(2)

display(tabela_tipo_proposta_por_grupo)

ID_TP_PRP,0,1,2
PERFIL_GENERO_GRUPO_CLIENTES,,,
Apenas Homens,73.68,24.36,1.96
Apenas Mulheres,69.20,29.33,1.47
Misto,73.55,25.00,1.45


O grupo exclusivo de mulheres, possui uma quantidade maior de propostas do tipo renegociadas (Tipo 1) do que os demais grupos. Mas também uma taxa de propostas normais (Tipo 0) e refinanciadas (Tipo 2) relativamente menor que a dos grupos de clientes exclusivamente masculinos. No próximo notebook, posso realizar uma análise de Fairness a respeito disso, pois acho um pouco curioso que a taxa de propostas normais seja inferior considerando que o grupo de apenas mulheres possui muito mais propostas que de apenas homens.

#### Análise sobre valores solicitados na proposta

In [9]:
subsets = {
    "Mulheres": df_apenas_mulheres,
    "Homens": df_apenas_homens,
    "Mistos": df_misto,
}

In [10]:
estatisticas = []

for nome, sub_df in subsets.items():
    s = sub_df["VR_SOL_PRP"]
    estatisticas.append(
        {
            "Subgrupo": nome,
            "Contratos": len(s),
            "Média": s.mean(),
            "Mediana": s.median(),
            "Máximo": s.max(),
        }
    )

df_valores_proposto_por_grupo = pd.DataFrame(estatisticas)
df_valores_proposto_por_grupo

,Subgrupo,Contratos,Média,Mediana,Máximo
0,Mulheres,83725,11284.469585,8365.0,952346.0
1,Homens,19974,12989.457662,9390.0,146790.0
2,Mistos,177837,12873.001217,9330.0,327460.0


A mediana solicitada pelas mulheres (R$ 8.365,00) é menor do que a dos homens (R$ 9.390,00) e dos clientes mistos em gênero (R$ 9.330,00). A média segue a mesma tendência: mulheres pedem em média R$ 11.284, enquanto homens e mistos pedem mais de R$ 12.989. Vejo que há dois pontos interessantes aqui: 

1. Medidas de posição são maiores para grupos apenas de homens, mesmo que eles estejam em uma quantidade significativamente menor. 
2. Mesmo com homens em média realizando empréstimo de valores maiores, eles possuem maior quantidade de contratos do tipo "Normal" e mulheres possuem mais propostas renegociadas, apesar de solicitarem valores menores, em geral.  

Posso como próximos passos verificar se essa diferença possui alguma relação com a natureza de negócio do empreendimento dos clientes.

In [11]:
estatistica_per_capita = []

for nome, sub_df in subsets.items():
    sub_df["VR_SOL_PRP_PER_CAPITA"] = sub_df["VR_SOL_PRP"] / sub_df["QT_CLI"]

    estatistica_per_capita.append(
        {
            "Subgrupo": nome,
            "Contratos": len(sub_df["VR_SOL_PRP_PER_CAPITA"]),
            "Média Per Capita": sub_df["VR_SOL_PRP_PER_CAPITA"].mean(),
            "Mediana Per Capita": sub_df["VR_SOL_PRP_PER_CAPITA"].median(),
        }
    )

df_estat_per_capita = pd.DataFrame(estatistica_per_capita)
df_estat_per_capita

/tmp/ipykernel_21420/1763304335.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub_df["VR_SOL_PRP_PER_CAPITA"] = sub_df["VR_SOL_PRP"] / sub_df["QT_CLI"]
/tmp/ipykernel_21420/1763304335.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub_df["VR_SOL_PRP_PER_CAPITA"] = sub_df["VR_SOL_PRP"] / sub_df["QT_CLI"]
/tmp/ipykernel_21420/1763304335.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at o

,Subgrupo,Contratos,Média Per Capita,Mediana Per Capita
0,Mulheres,83725,2918.978198,2205.000000
1,Homens,19974,3667.121420,2757.666667
2,Mistos,177837,3081.131426,2277.500000


Ao observar o valor **solicitado** per capita à quantidade de clientes do contrato, tanto a média quanto a mediana do grupo exclusivamente formado por homens possui valor maior do que o grupo exclusivamente de mulheres e o grupo misto. Vejo com isso um indício de que homens possuem maior apetite ao risco.

#### Análise sobre valores aprovados

In [12]:
estatisticas = []

for nome, sub_df in subsets.items():
    s = sub_df["VR_APR_PRP"]
    estatisticas.append(
        {
            "Subgrupo": nome,
            "Contratos": len(s),
            "Média": s.mean(),
            "Mediana": s.median(),
            "Máximo": s.max(),
        }
    )

df_valores_proposto_por_grupo = pd.DataFrame(estatisticas)
df_valores_proposto_por_grupo

,Subgrupo,Contratos,Média,Mediana,Máximo
0,Mulheres,83725,10334.217467,7730.0,94855.0
1,Homens,19974,12049.951110,8800.0,84182.0
2,Mistos,177837,11818.100643,8520.0,118675.0


In [13]:
estatistica_per_capita = []

for nome, sub_df in subsets.items():
    sub_df["VR_APR_PER_CAPITA"] = sub_df["VR_APR_PRP"] / sub_df["QT_CLI"]

    estatistica_per_capita.append(
        {
            "Subgrupo": nome,
            "Contratos": len(sub_df["VR_APR_PER_CAPITA"]),
            "Média Per Capita": sub_df["VR_APR_PER_CAPITA"].mean(),
            "Mediana Per Capita": sub_df["VR_APR_PER_CAPITA"].median(),
            "Valor Máx Per Capita": sub_df["VR_APR_PER_CAPITA"].max(),
        }
    )

df_estat_per_capita = pd.DataFrame(estatistica_per_capita)
df_estat_per_capita

/tmp/ipykernel_21420/1548504661.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub_df["VR_APR_PER_CAPITA"] = sub_df["VR_APR_PRP"] / sub_df["QT_CLI"]
/tmp/ipykernel_21420/1548504661.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  sub_df["VR_APR_PER_CAPITA"] = sub_df["VR_APR_PRP"] / sub_df["QT_CLI"]
/tmp/ipykernel_21420/1548504661.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once usin

,Subgrupo,Contratos,Média Per Capita,Mediana Per Capita,Valor Máx Per Capita
0,Mulheres,83725,2675.228255,2091.5,14147.0
1,Homens,19974,3401.184880,2575.0,14260.0
2,Mistos,177837,2828.859602,2100.0,15000.0


Grupos exclusivos de mulheres possuem uma mediana inferior a grupos exclusivos de homens e grupos mistos no que diz respeito ao valor aprovado de crédito, mesmo possuindo mais contratos e um valor máximo aprovado superior à grupos exclusivos de homens. Em uma análise per capita, vejo que o grupo de homens ainda possui um valor aprovado (tanto em média como em mediana) substancialmente superior ao grupo de mulheres e ao grupo misto em gênero e valor máximo per capita sendo superior, ao grupo exclusivo de mulheres mesmo tendo um valor máximo de contrato inferior ao grupo exclusivo de mulheres

#### Análise da relação entre grupos e a taxa do valor aprovado de proposta em relação ao solicitado

Tendo em vista que não há nenhuma relação significativa entre o gênero dos grupos e o valor solicitado de empréstimo, podemos analisar se há alguma associação entre o gênero dos clientes dos grupo e a taxa de crédito concedido pelo banco após análise. Ou seja, a razão entre o valor aprovado e o solicitado.

In [14]:
df_copy = df.copy()

In [15]:
# Porcentagem do valor aprovado em relação ao valor solicitado
df_copy['PRT_VLR_APR_SOL'] = (df_copy['VR_APR_PRP']/df_copy['VR_SOL_PRP']) * 100

In [16]:
resumo_aprovacao = (
    df_copy.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['PRT_VLR_APR_SOL']
    .agg(['count', 'mean', 'median', 'std', 'max'])
    .round(2)
)

resumo_aprovacao

,count,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,,
Apenas Homens,19974,93.29,100.00,12.50,313.52
Apenas Mulheres,83725,92.30,100.00,12.83,664.38
Misto,177837,92.24,99.58,12.49,314.29


O valor máximo de cada grupo é superior a 1, ou seja, há propostas que o valor aprovado é o dobro e até o triplo do valor solicitado. Preciso investigar se essas propostas possuem alguma característica em comum ou se são ruídos de anotação.

Para simplificação da análise, vou selecionar apenas em que o valor é igual ou menor que 100.

In [17]:
df_taxa_aprovacao_limitada = df_copy[df_copy['PRT_VLR_APR_SOL'] <= 100]

resumo_aprovacao = (
    df_taxa_aprovacao_limitada.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['PRT_VLR_APR_SOL']
    .agg(['count', 'mean', 'median', 'std', 'max'])
    .round(2)
)

resumo_aprovacao

,count,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,,
Apenas Homens,19802,93.14,100.00,12.25,100.0
Apenas Mulheres,82898,92.14,100.00,12.53,100.0
Misto,176246,92.10,99.34,12.37,100.0


Com uma limitação para análise de contratos com uma taxa de aprovação de igual ou menor que 100%, a mediana se mantém a mesma porém a média dos grupos composto apenas por mulheres ainda é inferior aos homens e há um desvio padrão superior, possivelmente indicando maior inconsistência na aprovação dos valores propostos. Podemos investigar futuramente se há relação com alguma outra variável categórica.

#### Analise sobre o tempo máximo de atraso e inadimplência

Como as variáveis de taxa de valor aprovado em relação ao solicitado não possui associação com o perfil de gênero do grupo, vou investigar melhor os dados sobre tempo de atraso de pagamento pois essa variável pode justificar ou ajudar a entender como é o perfil de pagamento dos grupos. Essa variável indica o quantos dias de atraso teve a parcela que teve o maior atraso entre todas elas. Caso seu valor seja maior que trinta, temos certeza que em algum momento o contrato esteve em situação de inadimplência.

In [18]:
estatisticas_maximo_dias_em_atraso = []

for nome, sub_df in subsets.items():
    estatisticas_maximo_dias_em_atraso.append(
        {
            "Subgrupo": nome,
            "Contratos": len(sub_df["QT_DD_ACM_ATR_MAX"]),
            "Média": sub_df["QT_DD_ACM_ATR_MAX"].mean(),
            "Mediana": sub_df["QT_DD_ACM_ATR_MAX"].median(),
            "Q1": sub_df["QT_DD_ACM_ATR_MAX"].quantile(0.25),
            "Q3": sub_df["QT_DD_ACM_ATR_MAX"].quantile(0.75),
            "Contratos Inadimplentes": (sub_df["QT_DD_ACM_ATR_MAX"] > 30).sum(),
            "Máximo": sub_df["QT_DD_ACM_ATR_MAX"].max(),
        }
    )

df_valores_aprovado_por_grupo = pd.DataFrame(estatisticas_maximo_dias_em_atraso)
df_valores_aprovado_por_grupo

,Subgrupo,Contratos,Média,Mediana,Q1,Q3,Contratos Inadimplentes,Máximo
0,Mulheres,83725,3.571968,0.0,0.0,2.0,1991,177
1,Homens,19974,4.829228,0.0,0.0,3.0,735,280
2,Mistos,177837,3.959030,0.0,0.0,3.0,4921,321


A grande parte dos contratos não possui atraso, porém tendo em vista que a quantidade de contratos dos três grupos possui uma amplitude muito grande, e que eles possuem médias relativamente próximas, acho que pode ser melhor observar como os dados estão distribuidos. O naturalmente esperado nessa situação é que um grupo que possua muitos contratos, também possua muitos inadimplentes, independente do gênero.

In [19]:
estatisticas_maximo_dias_em_atraso = []

for nome, sub_df in subsets.items():
    estatisticas_maximo_dias_em_atraso.append(
        {
            "Subgrupo": nome,
            "Contratos": len(sub_df["QT_DD_ACM_ATR_MAX"]),
            "Média": sub_df["QT_DD_ACM_ATR_MAX"].mean(),
            "Variância": np.var(sub_df["QT_DD_ACM_ATR_MAX"]),
            "Desvio Padrão": np.std(sub_df["QT_DD_ACM_ATR_MAX"]),
        }
    )

df_valores_aprovado_por_grupo = pd.DataFrame(estatisticas_maximo_dias_em_atraso)
df_valores_aprovado_por_grupo

,Subgrupo,Contratos,Média,Variância,Desvio Padrão
0,Mulheres,83725,3.571968,159.609264,12.633656
1,Homens,19974,4.829228,272.863147,16.518570
2,Mistos,177837,3.959030,178.627538,13.365161


Apesar das médias relativamente próximas, os desvios-padrão são sempre maiores que a própria média com o grupo de homens tendo a maior disperção do dia máximo de atraso entre os grupos indicando inconsistência/não-padronização no comportamento. Apesar disso, os dados indicam que 75% (Q3) dos contratos possui como maior atraso, entre 2 e 3 dias, não caracterizando uma situação de inadimplência. 

Vou então analisar apenas os contratos que em algum momento estiveram em situação de inadimplência. 

In [20]:
resultados = []

for grupo, sub_df in subsets.items():
    total = len(sub_df)
    inadimplentes = (sub_df['QT_DD_ACM_ATR_MAX'] > 30).sum()
    taxa = (inadimplentes / total) * 100

    resultados.append({
        'Grupo': grupo,
        'Total de contratos': total,
        'Contratos inadimplentes': inadimplentes,
        'Taxa de inadimplência (%)': taxa
    })

tabela_inadimplencia = pd.DataFrame(resultados)

tabela_inadimplencia['Taxa de inadimplência (%)'] = (tabela_inadimplencia['Taxa de inadimplência (%)'].round(2))

tabela_inadimplencia

,Grupo,Total de contratos,Contratos inadimplentes,Taxa de inadimplência (%)
0,Mulheres,83725,1991,2.38
1,Homens,19974,735,3.68
2,Mistos,177837,4921,2.77


Apesar do grupo de mulheres possuir mais contratos inadimplentes, a taxa de inadimplência do seu grupo é inferior à taxa do grupo de homens e do grupo misto. O que isso quer dizer? Considerando também que grupos exclusivamente composto por homens possuem (1) maior taxa de aprovação em relação ao valor proposto, (2) maior valor médio proposto per capita, (3) menor taxa de contrato renegociado. Me parece que grupos exclusivos de mulheres são mais confiaveis

In [21]:
subset_adimplente = df_copy[df_copy['QT_DD_ACM_ATR_MAX'] <= 30]
subset_inadimplentes = df_copy[df_copy['QT_DD_ACM_ATR_MAX'] > 30]

- Análise de contratos inadimplentes

In [ ]:
agg_tempo_inadimplencia = subset_inadimplentes.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['QT_DD_ACM_ATR_MAX'].agg(['mean', 'median', 'std', 'max']).round(2)
agg_tempo_inadimplencia

,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,
Apenas Homens,76.23,73.0,39.36,280
Apenas Mulheres,69.69,55.0,37.61,177
Misto,68.08,52.0,38.25,321


Mesmo inadimplentes, grupos exclusivamente femininos costumam quitar seus créditos com maior rapidez do que grupos exclusivamente masculinos

In [36]:
tabela_tipo_proposta_por_grupo = (pd.crosstab(subset_inadimplentes['PERFIL_GENERO_GRUPO_CLIENTES'], subset_inadimplentes['ID_TP_PRP'], normalize='index')*100).round(2)
tabela_tipo_proposta_por_grupo

ID_TP_PRP,0,1,2
PERFIL_GENERO_GRUPO_CLIENTES,,,
Apenas Homens,57.01,19.18,23.81
Apenas Mulheres,55.35,20.74,23.91
Misto,61.78,16.40,21.82


In [23]:
df_taxa_aprovacao_limitada = subset_inadimplentes[subset_inadimplentes['PRT_VLR_APR_SOL'] <= 100]

resumo_aprovacao = (
    df_taxa_aprovacao_limitada.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['PRT_VLR_APR_SOL']
    .agg(['count', 'mean', 'median', 'std', 'max'])
    .round(2)
)

resumo_aprovacao

,count,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,,
Apenas Homens,726,85.62,91.99,17.66,100.0
Apenas Mulheres,1973,84.82,89.75,17.38,100.0
Misto,4880,84.52,89.06,16.96,100.0


Dentre o subconjunto de contratos que se tornaram inadimplentes em algum momento, o grupo exclusivo de mulheres ainda possui a menor taxa de contratos do tipo normal e a maior taxa do tipo renegociada apesar de possuir a menor taxa de contratos em inadimplência entre os três grupos. Enquanto isso, o grupo exclusivo de homens possui a maior média, mediana e desvio padrão de taxa de valor aprovado. Eu entendo que a ocorrência da inadimplência não é premeditada mas é engraçado que as mulheres em geral tem um valor aprovado menor que o solicitado mas costumam possuir a menor taxa inadimplência, sendo assim possivelmente mais confiaveis. O que isso está querendo nos dizer? 

In [32]:
agg_prestacoes_aprovadas = subset_adimplente.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['QT_PRS_APR'].agg(['mean', 'median', 'std', 'max']).round(2)
agg_prestacoes_aprovadas

,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,
Apenas Homens,5.66,6.0,0.98,14
Apenas Mulheres,5.58,6.0,0.80,17
Misto,5.58,6.0,0.82,17


In [ ]:
agg_prestacoes_aprovadas = subset_inadimplentes.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['QT_PRS_APR'].agg(['mode', 'mean', 'median', 'std', 'max']).round(2)
agg_prestacoes_aprovadas

,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,
Apenas Homens,5.67,5.0,2.05,22
Apenas Mulheres,5.78,6.0,1.89,17
Misto,5.81,6.0,1.71,24


Nos subconjuntos de contratos adimplentes e inadimplentes, consigo observar que independente do gênero a quantidade de parcelas de cada grupo está em torno de 6 parcelas e um desvio padrão extramamente semelhante, variando cerca de 0.2 pontos percentuais. Ou seja, pelo menos nesse quesito há uma padronização seja devido à regras do banco ou por outro fator.

In [24]:
agg_valor_medio_prestacao = subset_adimplente.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['VR_MED_PRS'].agg(['mean', 'median', 'std', 'max']).round(2)
agg_valor_medio_prestacao

,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,
Apenas Homens,664.53,490.48,564.57,14491.60
Apenas Mulheres,524.85,395.92,423.76,11796.16
Misto,555.71,408.98,456.67,5889.14


In [25]:
agg_valor_medio_prestacao = subset_inadimplentes.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['VR_MED_PRS'].agg(['mean', 'median', 'std', 'max']).round(2)
agg_valor_medio_prestacao

,mean,median,std,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,
Apenas Homens,480.46,276.50,781.91,11670.85
Apenas Mulheres,323.01,199.47,576.75,11984.52
Misto,301.05,213.21,290.29,3790.91


Agora no que diz respeito ao valor médio das parcelas entre os clientes do contrato, o grupo exclusivo de homens possui uma mediana e desvio padrão superior ao grupo exclusivo de mulheres e ao grupo de gênero misto. Acredito que isso ocorra devido aos contratos exclusivamente masculinos possuirem, em sua mediana, 3 pessoas ao invés de 4 como os demais grupos e por este grupo não apenas solicitar mais crédito como também possuir em geral, maior valores de crédito aprovado e maior taxa de aprovação (em relação ao solicitado).   

### Dados de Juros

In [ ]:
agg_valor_taxa_juros_aprovada = subset_adimplente.groupby('PERFIL_GENERO_GRUPO_CLIENTES')['TX_JRS_APR'].agg(['mean', 'median', 'std', 'min', 'max']).round(2)
agg_valor_taxa_juros_aprovada

,mean,median,std,min,max
PERFIL_GENERO_GRUPO_CLIENTES,,,,,
Apenas Homens,1.96,2.3,0.42,1.06,2.3
Apenas Mulheres,1.94,2.3,0.44,1.06,2.3
Misto,1.98,2.3,0.43,0.95,2.3


Ao se observar a taxa de juros aprovada para cada um dos grupos, vejo que o valor da taxa é praticamente o mesmo para todos. Mesma mediana, mesmo valor máximo e mesmo valor mínimo para grupos exclusivamente masculinos e femininos. Minha interpretação é que apesar dos grupos exclusivamente femininos solicitarem valores proporcionalmente menores, terem relativamente uma aprovação menor ao solicitado, possuirem menor inadimplência e quando estão nessa situação, costumam realizar o pagamento mais rapidamente. 

Com isso, vejo que há diversos indicativos de grupos exclusivamente masculinos possuem um "tratamento diferente" em relação à grupos compostos exclusivamente por mulheres e grupos mistos. Digo tratamento diferente tendo em vista que possuem as mesmas condições de juros, quantidade de prestações aprovadas, periodicidade semelhante aos demais grupos apesar de possuírem comportamentos de maior risco ao banco.

Entendo que esses comportamentos ocorreram depois do contrato assinado, mas então posso investigar os dados passados que foram fornecidos pela pesquisa cadastral dos clientes que indicam a quantidade e a média de passagens negativas pelo SPC, Serada e pelo BNB.

### Dados de Pesquisa Cadastrais

### Dados de Seguro

### Dados de Natureza de Negócio